In [2]:
# =========================
# FINAL UPGRADE VERSION
# =========================

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers

# =========================
# DATASET
# =========================
text = """artificial intelligence systems learn patterns from data.
sequence models process information step by step.
recurrent neural networks are useful for sequence prediction.
lstm networks handle long term dependencies.

deep learning models improve sequence learning.
generative models create new samples from learned patterns.
language models predict the next word in a sentence.
sequence generation is used in chatbots and assistants.

machine learning helps computers learn automatically.
training data improves model accuracy.
neural networks simulate human brain structures.
optimization algorithms improve learning efficiency.

technology is transforming modern education.
online learning platforms use artificial intelligence.
students benefit from intelligent tutoring systems.
automation improves productivity and decision making."""

text = text * 50  # 🔥 more data

# =========================
# TOKENIZER (LSTM)
# =========================
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index) + 1

input_sequences = []
for line in text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_len = max(len(seq) for seq in input_sequences)

input_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    input_sequences, maxlen=max_len, padding='pre'
)

X_lstm = input_sequences[:, :-1]
y_lstm = to_categorical(input_sequences[:, -1], num_classes=total_words)

# =========================
# STRONGER LSTM MODEL
# =========================
lstm_model = tf.keras.Sequential([
    layers.Embedding(total_words, 128),
    layers.LSTM(150, return_sequences=True),
    layers.LSTM(150),
    layers.Dense(total_words, activation='softmax')
])

lstm_model.compile(loss='categorical_crossentropy', optimizer='adam')
lstm_model.fit(X_lstm, y_lstm, epochs=200, verbose=0)

# =========================
# TOP-K SAMPLING
# =========================
def top_k_sampling(preds, k=5, temperature=0.7):
    preds = np.log(preds + 1e-8) / temperature
    top_k = np.argsort(preds)[-k:]
    probs = np.exp(preds[top_k])
    probs = probs / np.sum(probs)
    return np.random.choice(top_k, p=probs)

# =========================
# LSTM GENERATION
# =========================
def generate_lstm(seed, n=15):
    for _ in range(n):
        token_list = tokenizer.texts_to_sequences([seed])[0]
        token_list = tf.keras.preprocessing.sequence.pad_sequences(
            [token_list], maxlen=max_len-1, padding='pre'
        )

        preds = lstm_model.predict(token_list, verbose=0)[0]
        idx = top_k_sampling(preds, k=5)

        for word, index in tokenizer.word_index.items():
            if index == idx:
                if word not in seed.split()[-3:]:
                    seed += " " + word
                break
    return seed

print("\n🔥 LSTM Output:")
print(generate_lstm("artificial intelligence", 15))


# =========================
# TRANSFORMER
# =========================
vectorizer = layers.TextVectorization(output_mode='int')
vectorizer.adapt([text])

vocab_size = len(vectorizer.get_vocabulary())
sequences = vectorizer([text])[0].numpy()

seq_length = 8  # more context
X, y = [], []

for i in range(len(sequences) - seq_length):
    X.append(sequences[i:i+seq_length])
    y.append(sequences[i+seq_length])

X = np.array(X)
y = np.array(y)

# Positional Encoding
class PositionalEmbedding(layers.Layer):
    def __init__(self, seq_len, vocab, dim):
        super().__init__()
        self.token = layers.Embedding(vocab, dim)
        self.pos = layers.Embedding(seq_len, dim)

    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        return self.token(x) + self.pos(positions)

# Transformer Block (deeper)
class TransformerBlock(layers.Layer):
    def __init__(self, dim, heads, ff):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=heads, key_dim=dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff, activation="relu"),
            layers.Dense(dim),
        ])
        self.norm1 = layers.LayerNormalization()
        self.norm2 = layers.LayerNormalization()

    def call(self, x):
        attn = self.att(x, x)
        x = self.norm1(x + attn)
        return self.norm2(x + self.ffn(x))

# Model
inputs = layers.Input(shape=(seq_length,))
x = PositionalEmbedding(seq_length, vocab_size, 128)(inputs)
x = TransformerBlock(128, 4, 256)(x)
x = TransformerBlock(128, 4, 256)(x)  # 🔥 extra layer
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)

transformer_model = tf.keras.Model(inputs, outputs)
transformer_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

transformer_model.fit(X, y, epochs=80, verbose=0)

# =========================
# TRANSFORMER GENERATION
# =========================
def generate_transformer(seed, n=15):
    for _ in range(n):
        tokenized = vectorizer([seed])[0].numpy()[-seq_length:]
        tokenized = np.pad(tokenized, (seq_length-len(tokenized), 0))

        preds = transformer_model.predict(tokenized.reshape(1, -1), verbose=0)[0]
        idx = top_k_sampling(preds, k=5, temperature=0.8)

        word = vectorizer.get_vocabulary()[idx]

        if word not in seed.split()[-3:]:
            seed += " " + word

    return seed

print("\n🚀 Transformer Output:")
print(generate_transformer("machine learning", 15))



🔥 LSTM Output:
artificial intelligence systems learn patterns from data systems patterns

🚀 Transformer Output:
machine learning learn models a learning generative data sequence
